In [2]:

from pathlib import Path
import json

import pandas as pd

def find_project_root(start: Path) -> Path:
    """Walk up from the current folder until we find the repo root."""
    for candidate in (start, *start.parents):
        if (candidate / "data" / "input").is_dir():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "structured"
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"

print("EXTRACTION VALIDATION")
print("=" * 60)

# 1. Check whether the main output files exist.
for filename in ["document.md", "document_structure.json", "extraction_summary.json"]:
    path = OUTPUT_DIR / filename
    print(f"{filename}: {'FOUND' if path.exists() else 'MISSING'}")

# 2. Inspect every extracted table.
print("\nTABLE SUMMARY")
print("-" * 60)

table_reports = []

for csv_path in sorted(TABLES_DIR.glob("table_*.csv")):
    df = pd.read_csv(csv_path, dtype=str).fillna("")

    report = {
        "file": csv_path.name,
        "rows": len(df),
        "columns": len(df.columns),
        "empty_cells": int((df == "").sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
    }
    table_reports.append(report)

    print(
        f"\n{csv_path.name}: {report['rows']} rows x "
        f"{report['columns']} columns"
    )
    print(f"Empty cells: {report['empty_cells']}")
    print(f"Duplicate rows: {report['duplicate_rows']}")
    print(df.head(5).to_string(index=False))

# 3. Check saved figure images.
figure_files = sorted(FIGURES_DIR.glob("figure_*.png"))

print("\nFIGURE SUMMARY")
print("-" * 60)
print(f"Saved figure images: {len(figure_files)}")

for figure_path in figure_files:
    print(f"- {figure_path.name} ({figure_path.stat().st_size:,} bytes)")

# 4. Save the validation report.
report_path = OUTPUT_DIR / "validation_report.json"
report_path.write_text(
    json.dumps(
        {
            "table_count": len(table_reports),
            "tables": table_reports,
            "saved_figure_count": len(figure_files),
        },
        indent=2,
    ),
    encoding="utf-8",
)

print(f"\nValidation report saved to: {report_path}")


EXTRACTION VALIDATION
document.md: FOUND
document_structure.json: FOUND
extraction_summary.json: FOUND

TABLE SUMMARY
------------------------------------------------------------

table_001.csv: 2 rows x 2 columns
Empty cells: 0
Duplicate rows: 0
                      0                   1
       BSE CODE: 500400 NSE CODE: TATAPOWER
Bloomberg CODE: TPWR:IN      SENSEX: 67,127

table_002.csv: 23 rows x 4 columns
Empty cells: 0
Duplicate rows: 0
                      0                       1                       2            3
           Company Data            Company Data            Company Data Company Data
        Market Cap (cr)         Market Cap (cr)         Market Cap (cr)   Rs. 86,856
  Enterprise Value (cr)   Enterprise Value (cr)   Enterprise Value (cr)  Rs. 131,339
Outstanding Shares (cr) Outstanding Shares (cr) Outstanding Shares (cr)        319.6
             Free Float              Free Float              Free Float        53.0%

table_003.csv: 13 rows x 4 columns
Empty 

In [3]:

print("\nDETAILED TABLE INSPECTION")
print("=" * 60)

for table_number in range(1, 6):
    csv_path = TABLES_DIR / f"table_{table_number:03}.csv"

    if not csv_path.exists():
        continue

    df = pd.read_csv(csv_path, dtype=str).fillna("")

    print(f"\n{'=' * 20} TABLE {table_number} {'=' * 20}")
    print(df.to_string(index=False))

print("\nMARKDOWN PREVIEW")
print("=" * 60)

markdown_path = OUTPUT_DIR / "document.md"
markdown = markdown_path.read_text(encoding="utf-8")

print(markdown[:6000])



DETAILED TABLE INSPECTION

==================== TABLE 1 ====================
                      0                   1
       BSE CODE: 500400 NSE CODE: TATAPOWER
Bloomberg CODE: TPWR:IN      SENSEX: 67,127

==================== TABLE 2 ====================
                      0                       1                       2            3
           Company Data            Company Data            Company Data Company Data
        Market Cap (cr)         Market Cap (cr)         Market Cap (cr)   Rs. 86,856
  Enterprise Value (cr)   Enterprise Value (cr)   Enterprise Value (cr)  Rs. 131,339
Outstanding Shares (cr) Outstanding Shares (cr) Outstanding Shares (cr)        319.6
             Free Float              Free Float              Free Float        53.0%
         Dividend Yield          Dividend Yield          Dividend Yield         0.7%
           52 week high            52 week high            52 week high      Rs. 277
            52 week low             52 week low            